In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
import textwrap
import matplotlib.colors as colors

In [2]:
# read in tc_basins file so we can filter to a specific ocean basin
polygons_dict = {}

with open("tc_basins_NAtl.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# read in tc_subbasins_NAtl file
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)
basins["geometry"] = basins["geometry"].apply(shift_lon)

In [4]:
## read in and filter our syclops data
# path to the classified dataset
ClassifiedData = r"datasets/SyCLoPS/SyCLoPS_classified_ERA5_1940_2024_v7.parquet"

# open the parquet format file (PyArrow package required)
df = pd.read_parquet(ClassifiedData)

# TC Nodes: filter to TCs only, tropical flag = 1 so we are only counting before they become extratropical
dftc = df[(df.Tropical_Flag==1) & ((df.Adjusted_Label=='TC') | (df.Adjusted_Label=='TD')) & ~(df['Track_Info'].str.contains('QS', case=False, na=False))]
# dftc = df[(df.Tropical_Flag==1) & (df.Adjusted_Label=='TC') & ~(df['Track_Info'].str.contains('QS', case=False, na=False))]


In [5]:
# sort by TID and ISOTIME
dftc = dftc.sort_values(['TID', 'ISOTIME'])

# convert lon to -180-180 from 0-360
dftc['LON'] = ((dftc['LON'] + 180) % 360) - 180


In [6]:
# first node: where the TC originates
tc_origin = (
    dftc
    .sort_values(by='ISOTIME')
    .groupby('TID', as_index=False)
    .head(1)
)

# last node: where the TC dissipates
TC_TIDs = tc_origin['TID']
tc_dissipate = (
    dftc[(dftc['TID'].isin(TC_TIDs)) & (dftc['Tropical_Flag'] == 1)]
    .sort_values(by='ISOTIME')
    .groupby('TID', as_index=False)
    .tail(1)
)

In [7]:
# convert LAT and LON to a new column Points which contains (lon, lat) and convert to a geo data frame so we can filter using polygons
tc_origin_points = gpd.GeoDataFrame(
    tc_origin, 
    geometry = gpd.points_from_xy(tc_origin.LON, tc_origin.LAT),
    crs = "EPSG:4326"
)

tc_dissipate_points = gpd.GeoDataFrame(
    tc_dissipate, 
    geometry = gpd.points_from_xy(tc_dissipate.LON, tc_dissipate.LAT),
    crs = "EPSG:4326"
)

In [8]:
# filter points to North Atlantic
tc_origin_filtered = gpd.sjoin(
    tc_origin_points,
    basins[basins["basin name"] == "N Atlantic"],
    how = "inner",
    predicate = "within"
)

tc_dissipate_filtered = gpd.sjoin(
    tc_dissipate_points,
    basins[basins["basin name"] == "N Atlantic"],
    how = "inner",
    predicate = "within"
)


In [ ]:
# merge start and end nodes on TID
tc_track = (
    tc_origin_filtered
    .merge(
        tc_dissipate_filtered,
        on="TID",
        suffixes=("_start", "_end")
    )
)

# print(tc_track.columns)

Index(['TID', 'LON_start', 'LAT_start', 'ISOTIME_start', 'MSLP_start',
       'WS_start', 'ZS_start', 'Short_Label_start', 'Adjusted_Label_start',
       'Tropical_Flag_start', 'Transition_Zone_start', 'Track_Info_start',
       'LPSAREA_start', 'i_start', 'j_start', 'geometry_start',
       'index_right_start', 'basin name_start', 'LON_end', 'LAT_end',
       'ISOTIME_end', 'MSLP_end', 'WS_end', 'ZS_end', 'Short_Label_end',
       'Adjusted_Label_end', 'Tropical_Flag_end', 'Transition_Zone_end',
       'Track_Info_end', 'LPSAREA_end', 'i_end', 'j_end', 'geometry_end',
       'index_right_end', 'basin name_end'],
      dtype='str')


In [11]:
# add Year column so we can create a timeseries
tc_track['YEAR_start'] = tc_track['ISOTIME_start'].dt.year
tc_track['YEAR_end'] = tc_track['ISOTIME_end'].dt.year

# filter to only columns we need
tc_track = tc_track[['TID', 'LON_start', 'LAT_start', 'LON_end', 'LAT_end', 'YEAR_start']]


In [12]:
# join sub basin name for starting and ending points
start_gdf = gpd.GeoDataFrame(
    tc_track,
    geometry=gpd.points_from_xy(
        tc_track.LON_start,
        tc_track.LAT_start
    ),
    crs=sub_basins.crs
)

end_gdf = gpd.GeoDataFrame(
    tc_track,
    geometry=gpd.points_from_xy(
        tc_track.LON_end,
        tc_track.LAT_end
    ),
    crs=sub_basins.crs
)

start_join = gpd.sjoin(
    start_gdf,
    sub_basins[['sub_basin_name', 'geometry']],
    how='left',
    predicate='within'
)

end_join = gpd.sjoin(
    end_gdf,
    sub_basins[['sub_basin_name', 'geometry']],
    how='left',
    predicate='within'
)


In [13]:
tc_track['sub_basin_start'] = start_join['sub_basin_name']
tc_track['sub_basin_end'] = end_join['sub_basin_name']

# calc difference in lat/lon
tc_track['LON_diff_abs'] = (tc_track['LON_end'] - tc_track['LON_start']).abs()
tc_track['LAT_diff_abs'] = (tc_track['LAT_end'] - tc_track['LAT_start']).abs()

tc_track['LON_diff'] = tc_track['LON_end'] - tc_track['LON_start']
tc_track['LAT_diff'] = tc_track['LAT_end'] - tc_track['LAT_start']

In [15]:
# filter to gulfs
tc_track = tc_track[(tc_track['sub_basin_start'] == 'Gulf (A)') | (tc_track['sub_basin_start'] == 'Gulf (B)')]

print(tc_track)

         TID  LON_start  LAT_start  LON_end  LAT_end  YEAR_start  \
5       3859     -95.75      18.00   -95.75    18.00        1940   
6       3923     -95.00      31.75   -95.00    31.75        1940   
7       4246     -95.75      18.75   -98.75    22.75        1940   
8       4484     -95.75      26.00   -94.75    24.75        1940   
12      5175     -82.50      33.75   -92.50    35.50        1940   
...      ...        ...        ...      ...      ...         ...   
3569  735727     -91.75      18.50   -97.25    20.25        2024   
3582  737363     -95.25      27.75   -94.50    26.25        2024   
3584  737369     -91.25      18.75   -92.00    35.25        2024   
3599  738084     -96.00      21.00   -55.75    30.25        2024   
3601  738115     -85.50      26.75   -86.00    26.50        2024   

     sub_basin_start          sub_basin_end  LON_diff_abs  LAT_diff_abs  \
5           Gulf (B)               Gulf (B)          0.00          0.00   
6           Gulf (B)             